# Hypothesis Testing, Confidence Intervals and P-values

**Dataset:** `banking_operations.csv`  
**Tools:** pandas, NumPy, Matplotlib and topic-specific statistical/ML functions  

This notebook explains the concept in simple terms and connects every calculation to banking operations.

## 1. Business question

Do Mobile Banking and Branch transactions have different average amounts? We use a two-sample test and a confidence interval to evaluate the difference.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations.csv")
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
from scipy import stats

groups = df[df["Channel"].isin(["Mobile Banking", "Branch"])].copy()
display(groups.groupby("Channel")["Amount"].agg(["count", "mean", "median", "std"]))

## 2. State the hypotheses

- **H0:** The two population mean transaction amounts are equal.
- **H1:** The two population mean transaction amounts are different.

The significance level is set to 0.05 before looking at the test result.

In [ ]:
alpha = 0.05
mobile_amounts = groups.loc[groups["Channel"] == "Mobile Banking", "Amount"]
branch_amounts = groups.loc[groups["Channel"] == "Branch", "Amount"]

test_result = stats.ttest_ind(mobile_amounts, branch_amounts, equal_var=False)
print(f"Welch t-statistic: {test_result.statistic:.3f}")
print(f"p-value: {test_result.pvalue:.4f}")

## 3. Interpret the p-value correctly

The p-value asks how unusual this result, or something more extreme, would be if H0 were true. It is **not** the probability that H0 is true and does not measure business importance.

In [ ]:
if test_result.pvalue <= alpha:
    decision = "Reject H0: the sample provides evidence of a mean difference."
else:
    decision = "Fail to reject H0: the sample does not provide enough evidence of a mean difference."
print(decision)

## 4. Confidence interval for the difference in means

A confidence interval shows the size and uncertainty of the estimated difference. We bootstrap the two groups with pandas sampling.

In [ ]:
bootstrap_differences = []
for seed in range(3000):
    mobile_sample = mobile_amounts.sample(len(mobile_amounts), replace=True, random_state=seed)
    branch_sample = branch_amounts.sample(len(branch_amounts), replace=True, random_state=seed + 5000)
    bootstrap_differences.append(mobile_sample.mean() - branch_sample.mean())

bootstrap_differences = pd.Series(bootstrap_differences, name="Mean difference")
lower, upper = bootstrap_differences.quantile([0.025, 0.975])
observed_difference = mobile_amounts.mean() - branch_amounts.mean()

display(pd.Series({
    "Observed Mean Difference (Mobile - Branch)": observed_difference,
    "95% CI Lower": lower,
    "95% CI Upper": upper
}).to_frame("Value"))

In [ ]:
bootstrap_differences.plot(kind="hist", bins=30, edgecolor="black", color="#56CCF2", figsize=(8, 4))
plt.axvline(0, color="red", linestyle="--", label="No difference")
plt.axvline(lower, color="green", linestyle=":", label="95% limits")
plt.axvline(upper, color="green", linestyle=":")
plt.title("Bootstrap Distribution of Mean Difference")
plt.xlabel("Mobile Mean - Branch Mean")
plt.legend()
plt.show()

## 5. Optional categorical test: Channel and Status

A chi-square test checks whether two categorical variables are associated.

In [ ]:
contingency = pd.crosstab(df["Channel"].fillna("Unknown"), df["Status"])
chi2, chi_p, degrees_freedom, expected = stats.chi2_contingency(contingency)
display(contingency)
print(f"Chi-square statistic: {chi2:.3f}")
print(f"p-value: {chi_p:.4f}")
print(f"Degrees of freedom: {degrees_freedom}")

## Banking interpretation and cautions

Always report the estimate, confidence interval, p-value, sample sizes and practical importance together. Failing to reject H0 does not prove equality. A small p-value does not prove causation, and small category counts can make statistical results unstable.